# Forward Feature Selection

Practice activity from Microsoft *Foundations of AI and Machine Learning* — Module: AI/ML Algorithms and Techniques.

Goal: build a forward-selection loop that greedily adds features by R-squared improvement, then evaluate the chosen feature set on a held-out split.

## 1. Imports & data

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

data = {
    'StudyHours':    [1,  2,  3,  4,  5,  6,  7,  8,  9, 10],
    'PrevExamScore': [30, 40, 45, 50, 60, 65, 70, 75, 80, 85],
    'Pass':          [0,  0,  0,  0,  0,  1,  1,  1,  1,  1],
}
df = pd.DataFrame(data)

X = df[['StudyHours', 'PrevExamScore']]
y = df['Pass']
df

,StudyHours,PrevExamScore,Pass
0,1,30,0
1,2,40,0
2,3,45,0
3,4,50,0
4,5,60,0
5,6,65,1
6,7,70,1
7,8,75,1
8,9,80,1
9,10,85,1


## 2. Implementing forward selection

Forward selection starts with **no features** and adds them one at a time. In each round we try every remaining feature, fit a linear regression on `selected + [candidate]`, score it with R-squared on a held-out test split, and keep the candidate with the highest score. We stop as soon as no remaining feature improves the score.

In [2]:
def forward_selection(X, y):
    remaining_features = set(X.columns)
    selected_features = []
    best_score = 0.0

    while remaining_features:
        scores_this_round = []

        for feature in remaining_features:
            trial = selected_features + [feature]
            X_train, X_test, y_train, y_test = train_test_split(
                X[trial], y, test_size=0.2, random_state=42
            )
            model = LinearRegression()
            model.fit(X_train, y_train)
            score = r2_score(y_test, model.predict(X_test))
            scores_this_round.append((score, feature))

        scores_this_round.sort(reverse=True)
        top_score, top_feature = scores_this_round[0]

        if top_score > best_score:
            selected_features.append(top_feature)
            remaining_features.remove(top_feature)
            best_score = top_score
            print(f'  Added {top_feature!r:18s}  R-squared = {top_score:.4f}')
        else:
            break

    return selected_features, best_score

print('Running forward selection...')
best_features, best_score = forward_selection(X, y)
print(f'\nSelected features: {best_features}')
print(f'Best R-squared during selection: {best_score:.4f}')

Running forward selection...
  Added 'PrevExamScore'     R-squared = 1.0000

Selected features: ['PrevExamScore']
Best R-squared during selection: 1.0000


## 3. Results analysis

On this dataset `PrevExamScore` is selected first and `StudyHours` adds nothing on top of it (in fact it would lower R² on this split), so the algorithm stops with a single feature.

## 4. Evaluating the final model

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X[best_features], y, test_size=0.2, random_state=42
)
final_model = LinearRegression()
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
final_r2 = r2_score(y_test, y_pred)

print(f'Final R-squared with selected features: {final_r2:.4f}')
print(f'Coefficients: {dict(zip(best_features, final_model.coef_))}')
print(f'Intercept:    {final_model.intercept_:.4f}')

Final R-squared with selected features: 1.0000
Coefficients: {'PrevExamScore': np.float64(0.024999999999999994)}
Intercept:    -1.0000


## Key takeaways

- Forward selection is **greedy**: at each step it picks the feature whose marginal contribution is largest given what's already selected. It's fast but can miss combinations that only shine together.
- `StudyHours` and `PrevExamScore` are highly correlated, so once one is selected the other contributes very little extra R² — a classic redundancy signal.
- The stopping rule used here ("no improvement") is the simplest variant. Production code usually uses cross-validated R², AIC/BIC, or statistical significance tests instead of a single split.
- Forward selection is most useful when you have many candidate features and you want an interpretable subset; on datasets with thousands of features, LASSO often gets you to a similar place faster.